# 격자 기둥으로 곡면 아래 부피 구하기 — 강의용 Colab 데모

**확률통계 · Topic 11 · Monte Carlo and Bootstrap**

슬라이드 12~13에서 1차원 구간을 조각내어 **넓이**를 더했다. 같은 일을 2차원 바닥에서 하면
곡면 아래 **부피**가 된다. 바닥 $[-1, 1]^2$ 를 $n \times n$ 칸으로 나누고, 칸마다 기둥을 세워 부피를 더한다.

$$V = \iint_{[-1,1]^2} e^{-(x^2+y^2)}\,dx\,dy \;\approx\; \sum_{i=1}^{n}\sum_{j=1}^{n} f(x_i, y_j)\, h^2, \qquad h = \frac{2}{n}$$

- 기둥 높이는 **칸 중심의 높이** $f(x_i, y_j)$ 다 (중점 규칙). 오차는 사다리꼴과 같은 $h^2$ 차수다.
- 함수 $e^{-(x^2+y^2)} = e^{-x^2} \cdot e^{-y^2}$ 는 슬라이드 14 예제 $e^{-x^2}$ 의 2차원판이다.
  그래서 참값이 $\left(\int_{-1}^{1} e^{-x^2} dx\right)^2 = (\sqrt{\pi}\,\mathrm{erf}(1))^2$ 로 정확히 나온다.

| 셀 | 내용 |
|---|---|
| ① | 설정 — 함수, 참값, 기둥 부피 합 계산 |
| ② | 한 장으로 보기 — $n$ = 2, 4, 8, 16 |
| ③ | ⭐ **애니메이션** — 기둥이 잘게 쪼개지며 부피 합이 참값으로 |
| ④ | 수렴 표 — $n$ 을 2배 하면 오차가 약 ¼ |
| ⑤ | 같은 점 수의 몬테카를로와 비교 (슬라이드 16) |

**① 을 먼저 실행한 뒤 나머지를 순서대로 실행한다.** ③ 은 프레임을 그리느라 잠시 걸린다.

In [ ]:
# ① 설정 — 함수, 참값, 기둥 부피 합
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from scipy import integrate
from scipy.special import erf

plt.rcParams["animation.embed_limit"] = 80      # 애니메이션을 노트북에 넣을 수 있는 최대 크기(MB)

def f(x, y):
    return np.exp(-(x**2 + y**2))

A, B = -1.0, 1.0                                 # 바닥은 [A, B] x [A, B]
EXACT = (np.sqrt(np.pi) * erf(1.0)) ** 2         # 참값 = (∫ e^{-x^2} dx)^2

# 참값을 scipy 의 2중 적분으로 한 번 더 확인한다
check, _ = integrate.dblquad(lambda y, x: f(x, y), A, B, A, B)
print(f"exact volume  = {EXACT:.6f}")
print(f"dblquad check = {check:.6f}")

def grid_volume(n):
    # n x n 칸의 중심에서 높이를 재고, 기둥 부피(높이 x 바닥 넓이 h^2)를 모두 더한다
    h = (B - A) / n
    c = A + h * (np.arange(n) + 0.5)             # 칸 중심의 좌표 n 개
    X, Y = np.meshgrid(c, c)                     # n x n 격자
    return (f(X, Y) * h * h).sum()

for n in [1, 2, 4, 8]:
    print(f"n = {n:>2}  ({n*n:>3} columns)  V_n = {grid_volume(n):.6f}")

## ② 한 장으로 보기

빨간 그물이 실제 곡면이고, 기둥이 근사다. $n$ 이 커질수록 기둥 윗면이 곡면에 달라붙는다.

In [ ]:
# ② n = 2, 4, 8, 16 을 한 그림에
def draw_columns(ax, n, elev=28, azim=-60):
    ax.computed_zorder = False                   # 깊이 자동 정렬을 끄고, 곡면을 항상 기둥 위에 그린다
    h = (B - A) / n
    left = A + h * np.arange(n)                  # 각 칸의 왼쪽 아래 모서리
    X0, Y0 = np.meshgrid(left, left)
    x0, y0 = X0.ravel(), Y0.ravel()
    top = f(x0 + h / 2, y0 + h / 2)              # 기둥 높이 = 칸 중심의 높이
    ax.bar3d(x0, y0, np.zeros_like(top), h, h, top,
             color=plt.cm.viridis(top), edgecolor="k", linewidth=0.2, alpha=0.85, shade=True, zorder=1)
    xs = np.linspace(A, B, 41)
    XS, YS = np.meshgrid(xs, xs)
    ax.plot_wireframe(XS, YS, f(XS, YS), color="crimson", linewidth=0.9, alpha=0.9,
                      rstride=4, cstride=4, zorder=5)
    ax.set_xlim(A, B); ax.set_ylim(A, B); ax.set_zlim(0, 1.05)
    ax.set_xticks([-1, 0, 1]); ax.set_yticks([-1, 0, 1]); ax.set_zticks([0, 0.5, 1])
    ax.view_init(elev=elev, azim=azim)

fig = plt.figure(figsize=(13, 3.6))
for k, n in enumerate([2, 4, 8, 16]):
    ax = fig.add_subplot(1, 4, k + 1, projection="3d")
    draw_columns(ax, n)
    ax.set_title(f"n = {n}:  V = {grid_volume(n):.4f}", fontsize=11)
fig.suptitle(f"Columns under z = exp(-(x^2+y^2)),  exact V = {EXACT:.4f}", fontsize=13)
plt.tight_layout()
plt.show()

## ③ ⭐ 애니메이션 — 기둥을 잘게 쪼갤수록 합이 적분으로

- **왼쪽**: $n \times n$ 기둥 (빨간 그물 = 실제 곡면). 프레임마다 $n$ 이 커지고 시점이 조금씩 돈다.
- **오른쪽**: 오차 $|V_n - V|$ 를 로그-로그로 그린다. 점선은 기울기 $-2$ ($\propto 1/n^2$) 기준선이다.

`NS` 목록을 바꾸면 다른 단계로 다시 그릴 수 있다. 숫자를 너무 키우면(예: 40 이상) 기둥이 많아져 느려진다.

In [ ]:
# ③ 애니메이션
NS = [1, 2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 24]   # 프레임마다 쓸 n
vols = [grid_volume(n) for n in NS]
errs = [abs(v - EXACT) for v in vols]

fig = plt.figure(figsize=(12.5, 5.2))
ax3 = fig.add_subplot(1, 2, 1, projection="3d")
ax2 = fig.add_subplot(1, 2, 2)

def frame(i):
    n = NS[i]
    ax3.cla()
    draw_columns(ax3, n, elev=28, azim=-60 + 3 * i)    # 프레임마다 시점을 3도씩 돌린다
    ax3.set_title(f"{n} x {n} = {n*n} column{'s' if n > 1 else ''}\nV_n = {vols[i]:.5f}   (exact {EXACT:.5f})", fontsize=12)

    ax2.cla()
    ref = errs[0] * (NS[0] / np.array(NS, dtype=float)) ** 2      # 1/n^2 기준선
    ax2.loglog(NS, ref, "--", color="gray", label="slope -2  (error ~ 1/n^2)")
    ax2.loglog(NS[: i + 1], errs[: i + 1], "o-", color="tab:blue", lw=2, label="|V_n - V|")
    ax2.loglog([n], [errs[i]], "o", color="crimson", ms=11)
    ax2.set_xlim(0.8, 30); ax2.set_ylim(1e-4, 5)
    ax2.set_xticks([1, 2, 4, 8, 16, 24]); ax2.set_xticklabels(["1", "2", "4", "8", "16", "24"])
    ax2.set_yticks([1e-4, 1e-3, 1e-2, 1e-1, 1]); ax2.set_yticklabels(["1e-4", "1e-3", "0.01", "0.1", "1"])
    ax2.minorticks_off()
    ax2.set_xlabel("n  (columns per axis)")
    ax2.set_ylabel("error  |V_n - V|")
    ax2.set_title(f"error = {errs[i]:.2e}", fontsize=12)
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc="upper right")

anim = animation.FuncAnimation(fig, frame, frames=len(NS), interval=900)
plt.close(fig)                       # 정지 그림이 한 번 더 나오지 않게
HTML(anim.to_jshtml())               # 재생 버튼이 달린 애니메이션

💾 파일로 저장하려면 아래 셀을 실행한다. GIF 는 Colab 왼쪽 파일 창에서 내려받을 수 있다.

In [ ]:
# (선택) GIF 로 저장 — Colab 왼쪽 '파일' 창에 생긴다
anim.save("grid_volume.gif", writer=animation.PillowWriter(fps=1.2))
print("saved grid_volume.gif")

## ④ 수렴 표 — $n$ 을 2배 하면 오차가 약 ¼

기둥 윗면은 평평하고 곡면은 휘어 있다. 그 **틈**이 오차다. 칸 폭 $h$ 를 반으로 줄이면 틈은 약 ¼ 이 된다.
슬라이드 13 의 오른쪽 그림(한 조각을 확대하면 틈이 오차)과 같은 이야기를 2차원에서 하는 것이다.

In [ ]:
# ④ n 을 2배씩 — 오차 비율이 4 에 가까워진다
prev = None
print(f"{'n':>4} {'columns':>8} {'V_n':>10} {'error':>10} {'ratio':>7}")
for n in [2, 4, 8, 16, 32, 64]:
    err = abs(grid_volume(n) - EXACT)
    ratio = f"{prev / err:7.2f}" if prev else "      -"
    print(f"{n:>4} {n*n:>8,} {grid_volume(n):>10.6f} {err:>10.2e} {ratio}")
    prev = err

## ⑤ 같은 점 수의 몬테카를로와 비교 — 슬라이드 16

칸이 $N = n^2$ 개면 $f$ 를 $N$ 번 계산한다. 같은 $N$ 번을 **무작위 점**에 쓰면 몬테카를로다.
2차원에서는 격자가 이긴다. 격자 오차는 $1/n^2 = 1/N$ 로 줄고, MC 표준오차는 $1/\sqrt{N}$ 로 줄기 때문이다.

⚠️ 그런데 $d$ 차원 격자는 축마다 $n$ 개면 칸이 $n^d$ 개다. 점 수 $N$ 을 고정하면 축당 점이 $N^{1/d}$ 개뿐이라
격자 오차는 $N^{-2/d}$ 로 느려지고, MC 는 여전히 $N^{-1/2}$ 다. $d = 4$ 에서 같아지고, 그보다 높으면 MC 가 이긴다.

In [ ]:
# ⑤ 같은 계산 횟수 N 에서 격자 vs MC
rng = np.random.default_rng(20260302)            # 과목 공통 시드
area = (B - A) ** 2
print(f"{'N = n^2':>9} {'grid error':>12} {'MC error':>12} {'MC std.err':>12}")
for n in [2, 4, 8, 16, 32, 64]:
    N = n * n
    g_err = abs(grid_volume(n) - EXACT)
    u = rng.uniform(A, B, size=(N, 2))           # 바닥에 무작위 점 N 개
    vals = f(u[:, 0], u[:, 1])
    mc = area * vals.mean()                      # 부피 = 바닥 넓이 x 평균 높이
    se = area * vals.std(ddof=1) / np.sqrt(N)    # CLT 표준오차 (슬라이드 11)
    print(f"{N:>9,} {g_err:>12.2e} {abs(mc - EXACT):>12.2e} {se:>12.2e}")

### 생각해 볼 것

1. `f` 를 `np.exp(-(x**2 + y**2)) * (1 + 0.5 * np.sin(6 * x))` 처럼 울퉁불퉁하게 바꾸면 ③ 의 오차 곡선은 어떻게 될까?
   참값은 `integrate.dblquad` 로 다시 구해야 한다.
2. ④ 에서 중점 대신 칸의 **왼쪽 아래 모서리** 높이를 쓰면 오차 비율이 4 대신 얼마가 될까? (힌트: 슬라이드 12 의 직사각형 $O(1/n)$)
3. 3차원 바닥 $[-1,1]^3$ 위의 4차원 "부피"를 같은 방식으로 구하면 $n = 24$ 일 때 칸이 몇 개가 될까?